# **Integration of NMR and LC-MS metabolomics datasets - Part 2**
## **Multivariate analysis integrating NMR and LC-MS data**

This tutorial notebook demonstrates the code provided with the protocol "Integration of NMR and LC-MS metabolomics datasets". Part 2 focuses on multivariate modelling severall data blocks together (NMR and LC-MS datasets) to create regression and classification models for predicting age and gender, respectively.

The datasets that will be inspected in this tutorial consist of an <sup>1</sup>H-NMR (CPMG), as well as 3 LC-MS datasets: HILIC positive (HPOS), Lipid positive (LPOS) and Lipid negative (LNEG). These datasets are part of the AddNeuroMed study (Lovestone *et al.* 2009), a dementia research cohort which contains NMR and LC-MS data.
All data matrices contain 573 samples that are in the same order.  

The code to perform the multiblock PLS models depends on the MAMSI package, which will be install with the code below:

In [ ]:
!pip install mamsi==1.0.8

Install the package to download data from Zenodo:

In [ ]:
!pip install zenodo-get

Read all the necessary packages and functions:

In [ ]:
import pandas as pd
import numpy as np
from zenodo_get import download
from zipfile import ZipFile
from sklearn.model_selection import train_test_split
from mamsi.mamsi_pls import MamsiPls
from mamsi.plots import plot_null_distribution
from matplotlib import pyplot as plt

Download all datasets and respective metadata from Zenodo:

In [ ]:
download(record_or_doi="22010261", output_dir="./", file_glob="*.zip")

Unzip the downloaded file:

In [ ]:
with ZipFile("alz_serum_data.zip", "r") as zip_ref:
    zip_ref.extractall("./")


Then read data:

In [ ]:
cpmg = pd.read_csv('alz_cpmg_features_serum.csv').add_prefix('CPMG_')
hpos = pd.read_csv('alz_hpos_serum.csv').add_prefix('HPOS_')
lpos = pd.read_csv('alz_lpos_serum.csv').add_prefix('LPOS_')
lneg = pd.read_csv('alz_lneg_serum.csv').add_prefix('LNEG_')

Prior to any modelling we must perform log-transformation. An ofset of 1 is added to all LC-MS datasets to avoid getting infinite values:

In [ ]:
cpmg = np.log(cpmg)
hpos = np.log1p(hpos)
lpos = np.log1p(lpos)
lneg = np.log1p(lneg)

### **1. MB-PLS regression of patient age**

In the first part of this notebook, we will create a multiblock PLS regression model to predict patient age using NMR and LC-MS data.

To create the regression model for the age we read the patient age from the metadata:

In [ ]:
metadata = pd.read_csv('alz_metadata_serum.csv')
y = metadata["Age"]

Prior to modelling the data, samples are going to be randomly split into training and test sets using a 90:10 split. The training data will be used for calculating the number of components and run the cross-validation, while the test set will be used to test the multivariate model. First the CPMG dataset and the Y data matrix are split into 90:10, then the indexes of the CPMG training and test matrices are used to split the remaining datasets, as the data matrices have the same samples:

In [ ]:
cpmg_train, cpmg_test, y_train, y_test = train_test_split(cpmg, y, test_size=0.1, random_state=42)

hpos_train = hpos.iloc[cpmg_train.index,:]
lpos_train = lpos.iloc[cpmg_train.index,:]
lneg_train = lneg.iloc[cpmg_train.index,:]

hpos_test = hpos.iloc[cpmg_test.index,:]
lpos_test = lpos.iloc[cpmg_test.index,:]
lneg_test = lneg.iloc[cpmg_test.index,:]


Next we fit the MB-PLS model with one component:

In [ ]:
mamsipls = MamsiPls(n_components=1)

mamsipls.fit([cpmg_train, hpos_train, lpos_train, lneg_train], y_train)

Then evaluate the optimal number of latent variables for the MB-PLS model for the regression of NMR+LC-MS against patient age:

In [ ]:
mamsipls.estimate_lv([cpmg_train, hpos_train, lpos_train, lneg_train], y_train, classification=False, metric='q2', max_components=10)

Two LVs are selected using the estimate_lv function. The function automatically refits the model with the estimated LVs.

The fitted model can now be evaluated by predicting the test set samples:

In [ ]:
predicted = mamsipls.evaluate_regression_model([cpmg_test, hpos_test, lpos_test, lneg_test], y_test.array)

### **2. Explore the differences between Female and Males using MB-PLS-DA**

The second part of this notebook demonstrates how to use multiblock PLS-DA to  to predict patient gender unsing NMR and LC-MS data and evaluate which varibles are important for the discriminating gender.

Read the "Gender" variable from the metadata and replace 'Female' and 'Male' by numeric values (1 and 0 respectively):

In [ ]:
metadata = pd.read_csv('alz_metadata_serum.csv')
y = metadata["Gender"].apply(lambda x: 1 if x == 'Female' else 0)

Train-test split (90:10):

In [ ]:
cpmg_train, cpmg_test, y_train, y_test = train_test_split(cpmg, y, test_size=0.1, random_state=42)

hpos_train = hpos.iloc[cpmg_train.index,:]
lpos_train = lpos.iloc[cpmg_train.index,:]
lneg_train = lneg.iloc[cpmg_train.index,:]

hpos_test = hpos.iloc[cpmg_test.index,:]
lpos_test = lpos.iloc[cpmg_test.index,:]
lneg_test = lneg.iloc[cpmg_test.index,:]

Model fitting:

In [ ]:
mamsipls = MamsiPls(n_components=1)

mamsipls.fit([cpmg_train, hpos_train, lpos_train, lneg_train], y_train)

Determine the optimal number of LVs:

In [ ]:
mamsipls.estimate_lv([cpmg_train, hpos_train, lpos_train, lneg_train], y_train, metric='auc')

Model classification performance using the test set:

In [ ]:
predicted = mamsipls.evaluate_class_model([cpmg_test, hpos_test, lpos_test, lneg_test], y_test.array)

Inspect importance of each block:

In [ ]:
mamsipls.block_importance(block_labels=["CPMG", "HPOS", "LPOS", "LNEG"])

Variable Importance to the Projection (VIP)

In [ ]:
mb_vip = mamsipls.mb_vip(plot=True, get_scores=True)
plt.vlines(x=cpmg.shape[1], ymin=0, ymax=18, colors='black', ls=':', lw=1)
plt.vlines(x=cpmg.shape[1]+hpos.shape[1], ymin=0, ymax=18, colors='black', ls=':', lw=1)
plt.vlines(x=cpmg.shape[1]+hpos.shape[1]+lpos.shape[1], ymin=0, ymax=18, colors='black', ls=':', lw=1)
plt.vlines(x=cpmg.shape[1]+hpos.shape[1]+lpos.shape[1]+lneg.shape[1], ymin=0, ymax=18, colors='black', ls=':', lw=1)

Permutation testing (Note: it might take up to 1 hour to run with 10000 permutations):

In [ ]:
p_vals = mamsipls.mb_vip_permtest([cpmg, hpos, lpos, lneg], y, n_permutations=10000, return_null_stats=True)

Plot the top 20 variables with highest VIP value:

In [ ]:
df = p_vals.sort_values(by=p_vals.columns[2], ascending=False)
df[['feature', 'p_value', 'observed_vip']].head(20)

It is also possible to plot the distributions of null (permuted) VIP values for a given feature and compare it against the observed VIP value:

In [ ]:
plot_null_distribution(p_vals, ind=56)

Save the full VIP data frame to file:

In [ ]:
df.to_csv('alz_FvsM_MB-PLSDA_VIP.csv', index=False)

**Reference**  
Lovestone S (2009) AddNeuroMed─The European Collaboration for the Discovery of Novel Biomarkers for Alzheimer's Disease. Ann NY Acad Sci 1180 (1), 36–46. doi: 10.1111/j.1749-6632.2009.05064.x